# MIMIC-IV Demo: dataset exploration and canonical ICU vital-sign extract

This notebook defines the source dataset used in the FAIRification project. It starts from the **MIMIC-IV Clinical Database Demo v2.2** files stored locally under `data/` and derives a small, reproducible resource containing three ICU vital signs:

- Heart rate (`itemid` 220045)
- Respiratory rate (`itemid` 220210)
- Pulse-oximetry oxygen saturation (`itemid` 220277)

The project resource contains observations from the **first four hours of every ICU stay represented in the demo**. The four-hour window is an **engineering scope boundary for the course project**, not a clinical claim about the importance of the first four ICU hours.

The final output of this notebook is written to `data/derived/mimic_icu_vitals_4h.csv`. Running the notebook from top to bottom should recreate the same file from the original MIMIC-IV Demo tables.

## 1. Setup and source files

Four source tables are needed for this derivation:

- `icu/chartevents.csv.gz`: recorded ICU observations
- `icu/icustays.csv.gz`: ICU stay identifiers, admission times, and first care units
- `icu/d_items.csv.gz`: dictionary describing MIMIC `itemid` codes
- `hosp/admissions.csv.gz`: hospital-admission identifiers and admission types

The helper below locates the repository root whether the notebook is started from the project root or from the `notebooks/` directory.

In [1]:
from pathlib import Path
import hashlib

import pandas as pd

pd.set_option("display.max_columns", 20)


def find_project_root() -> Path:
    """Find the repository root by looking for the expected MIMIC ICU directory."""
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "data" / "icu").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not find data/icu/. Run this notebook from inside the project repository "
        "and place the MIMIC-IV Demo files under data/icu/ and data/hosp/."
    )


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
ICU_DIR = DATA_DIR / "icu"
HOSP_DIR = DATA_DIR / "hosp"
DERIVED_DIR = DATA_DIR / "derived"

CHARTEVENTS_PATH = ICU_DIR / "chartevents.csv.gz"
ICUSTAYS_PATH = ICU_DIR / "icustays.csv.gz"
D_ITEMS_PATH = ICU_DIR / "d_items.csv.gz"
ADMISSIONS_PATH = HOSP_DIR / "admissions.csv.gz"

TARGET_ITEMIDS = [220045, 220210, 220277]
WINDOW_HOURS = 4

print(f"Project root: {PROJECT_ROOT}")
for path in (CHARTEVENTS_PATH, ICUSTAYS_PATH, D_ITEMS_PATH, ADMISSIONS_PATH):
    print(f"Source exists: {path.relative_to(PROJECT_ROOT)} -> {path.is_file()}")

Project root: /workspace/scratch/872a35266b67/FAIRifaction-MIMIC_IV
Source exists: data/icu/chartevents.csv.gz -> True
Source exists: data/icu/icustays.csv.gz -> True
Source exists: data/icu/d_items.csv.gz -> True
Source exists: data/hosp/admissions.csv.gz -> True


## 2. Load the relevant source columns

The full demo contains many more tables and variables than this project needs. We therefore load only the columns required to identify ICU stays, interpret the three measurements, and derive the bounded resource.

In [2]:
chartevents = pd.read_csv(
    CHARTEVENTS_PATH,
    usecols=[
        "subject_id",
        "hadm_id",
        "stay_id",
        "charttime",
        "itemid",
        "valuenum",
        "valueuom",
        "warning",
    ],
)

icustays = pd.read_csv(
    ICUSTAYS_PATH,
    usecols=["subject_id", "hadm_id", "stay_id", "intime", "first_careunit"],
)

d_items = pd.read_csv(
    D_ITEMS_PATH,
    usecols=[
        "itemid",
        "label",
        "abbreviation",
        "category",
        "unitname",
        "param_type",
    ],
)

admissions = pd.read_csv(
    ADMISSIONS_PATH,
    usecols=["subject_id", "hadm_id", "admission_type"],
)

source_summary = pd.DataFrame(
    {
        "table": ["chartevents", "icustays", "d_items", "admissions"],
        "rows_loaded": [len(chartevents), len(icustays), len(d_items), len(admissions)],
        "columns_loaded": [chartevents.shape[1], icustays.shape[1], d_items.shape[1], admissions.shape[1]],
    }
)
source_summary

Out[2]: 
         table  rows_loaded  columns_loaded
0  chartevents       668862               8
1     icustays          140               5
2      d_items         4014               6
3   admissions          275               3


## 3. Inspect the MIMIC measurement definitions

MIMIC uses local integer identifiers (`itemid`) for ICU charted measurements. The dictionary table gives those identifiers their source-specific meaning. This is important for the later FAIRification work: the source identifiers should be preserved, while semantic mappings can be added rather than replacing them.

In [3]:
target_items = (
    d_items[d_items["itemid"].isin(TARGET_ITEMIDS)]
    .sort_values("itemid")
    .reset_index(drop=True)
)

target_items

Out[3]: 
   itemid                        label abbreviation             category  \
0  220045                   Heart Rate           HR  Routine Vital Signs   
1  220210             Respiratory Rate           RR          Respiratory   
2  220277  O2 saturation pulseoxymetry         SpO2          Respiratory   

   unitname param_type  
0       bpm    Numeric  
1  insp/min    Numeric  
2         %    Numeric  


The three selected variables are all numeric vital-sign observations, but their meaning is currently encoded through MIMIC's local `itemid` vocabulary and source unit strings. This is one of the semantic-interoperability boundaries that the later FAIRification stages will address.

## 4. Define time relative to each ICU stay

The project scope is based on **elapsed time since ICU admission**, not clock time. Therefore, the correct reference is `icustays.intime`, joined to each observation using `subject_id`, `hadm_id`, and `stay_id`.

`charttime` is used as the observation time. `storetime` is not used for the window because it describes when a value was entered or validated rather than the time represented by the charted observation.

The same stay-level join carries `first_careunit`. This is the first care-unit type of the ICU stay; it does not establish the location of each individual observation.

In [4]:
chartevents["charttime"] = pd.to_datetime(chartevents["charttime"])
icustays["intime"] = pd.to_datetime(icustays["intime"])

selected_events = chartevents[chartevents["itemid"].isin(TARGET_ITEMIDS)].copy()

assert icustays["stay_id"].is_unique
selected_event_count = len(selected_events)

selected_events = selected_events.merge(
    icustays,
    on=["subject_id", "hadm_id", "stay_id"],
    how="left",
    validate="many_to_one",
)

assert len(selected_events) == selected_event_count
assert selected_events["intime"].notna().all(), "An observation has no matching ICU stay."

print(f"Selected HR/RR/SpO2 observations before time filtering: {len(selected_events):,}")
print(f"ICU stays represented before time filtering: {selected_events['stay_id'].nunique():,}")

Selected HR/RR/SpO2 observations before time filtering: 41,366
ICU stays represented before time filtering: 140


## 5. Explore candidate extraction windows

The course recommends a bounded resource of at most a few thousand records. Rather than choosing a window arbitrarily we opt for a subset that retains semantic meaning from extraction. We compare several elapsed-time windows and inspect how much of the demo population they retain.

In [5]:
def summarize_window(hours: int) -> dict:
    end_time = selected_events["intime"] + pd.Timedelta(hours=hours)
    subset = selected_events[
        (selected_events["charttime"] >= selected_events["intime"])
        & (selected_events["charttime"] < end_time)
    ]
    return {
        "window_hours": hours,
        "observations": len(subset),
        "patients": subset["subject_id"].nunique(),
        "hospital_admissions": subset["hadm_id"].nunique(),
        "icu_stays": subset["stay_id"].nunique(),
    }


window_summary = pd.DataFrame(
    [summarize_window(hours) for hours in [1, 2, 4, 6, 8, 24]]
)
window_summary

Out[5]: 
   window_hours  observations  patients  hospital_admissions  icu_stays
0             1           856        79                   99        109
1             2          1452        83                  107        119
2             4          2501        94                  121        133
3             6          3499        99                  127        139
4             8          4497       100                  128        140
5            24         11336       100                  128        140


## 6. Construct the canonical project resource

The interval is half-open: `[ICU intime, ICU intime + 4 hours)`. We then join the human-readable source label from `d_items`, while retaining the original `itemid`.

Following the TA's feedback about recurring categorical concepts, the canonical resource also includes:

- `first_careunit`, carried through the existing ICU-stay join;
- `admission_type`, joined from `admissions` on both `subject_id` and `hadm_id`.

The admission join is a **left, many-to-one join**: it must preserve every selected observation and cannot multiply rows. We check complete category coverage below. Original source labels are preserved without recoding.

The CSV repeats these contextual values on observation rows for convenience. In the later graph, `first_careunit` belongs to the **ICU stay**, and `admission_type` belongs to the **hospital admission**. All patient/admission/stay IDs and the warning flag remain. `admission_location` is deferred, and `last_careunit` is outside the agreed scope. No external vocabulary mappings are introduced here.


In [6]:
window_end = selected_events["intime"] + pd.Timedelta(hours=WINDOW_HOURS)

extract = selected_events[
    (selected_events["charttime"] >= selected_events["intime"])
    & (selected_events["charttime"] < window_end)
].copy()

extract = extract.merge(
    d_items[["itemid", "label"]],
    on="itemid",
    how="left",
    validate="many_to_one",
).rename(columns={"label": "measurement_label"})

# Preserve all observations while adding admission-level context.
assert not admissions.duplicated(["subject_id", "hadm_id"]).any()
before_admission_join = extract.copy()
extract = extract.merge(
    admissions,
    on=["subject_id", "hadm_id"],
    how="left",
    validate="many_to_one",
    sort=False,
)
assert len(extract) == len(before_admission_join)
pd.testing.assert_frame_equal(
    extract[before_admission_join.columns].reset_index(drop=True),
    before_admission_join.reset_index(drop=True),
)

extract = extract[
    [
        "subject_id",
        "hadm_id",
        "admission_type",
        "stay_id",
        "first_careunit",
        "charttime",
        "itemid",
        "measurement_label",
        "valuenum",
        "valueuom",
        "warning",
    ]
].sort_values(
    ["subject_id", "hadm_id", "stay_id", "charttime", "itemid", "valuenum"],
    kind="stable",
).reset_index(drop=True)

extract.head(10)

Out[6]: 
   subject_id   hadm_id admission_type   stay_id  \
0    10000032  29079034       EW EMER.  39553978   
1    10000032  29079034       EW EMER.  39553978   
2    10000032  29079034       EW EMER.  39553978   
3    10000032  29079034       EW EMER.  39553978   
4    10000032  29079034       EW EMER.  39553978   
5    10000032  29079034       EW EMER.  39553978   
6    10000032  29079034       EW EMER.  39553978   
7    10000032  29079034       EW EMER.  39553978   
8    10000032  29079034       EW EMER.  39553978   
9    10000032  29079034       EW EMER.  39553978   

                       first_careunit           charttime  itemid  \
0  Medical Intensive Care Unit (MICU) 2180-07-23 14:12:00  220045   
1  Medical Intensive Care Unit (MICU) 2180-07-23 14:12:00  220210   
2  Medical Intensive Care Unit (MICU) 2180-07-23 14:13:00  220277   
3  Medical Intensive Care Unit (MICU) 2180-07-23 14:30:00  220045   
4  Medical Intensive Care Unit (MICU) 2180-07-23 14:30:00  220210   
5  M

## 7. Validate the extract

Before treating this as the canonical project resource, we check coverage, measurement counts, missingness, duplicate observations, source warning flags, and observed value ranges. Unusual physiological values are not automatically removed: this notebook does not have enough evidence to classify them as errors.

Categorical summaries count distinct hospital admissions or ICU stays, rather than observation rows, so frequently charted stays do not inflate the category counts.

In [7]:
overall_summary = pd.Series(
    {
        "observations": len(extract),
        "patients": extract["subject_id"].nunique(),
        "hospital_admissions": extract["hadm_id"].nunique(),
        "icu_stays": extract["stay_id"].nunique(),
    },
    name="count",
)
overall_summary

Out[7]: 
observations           2501
patients                 94
hospital_admissions     121
icu_stays               133
Name: count, dtype: int64


In [8]:
measurement_summary = (
    extract.groupby(["itemid", "measurement_label", "valueuom"], dropna=False)["valuenum"]
    .agg(observations="size", minimum="min", median="median", maximum="max")
    .reset_index()
)
measurement_summary

Out[8]: 
   itemid            measurement_label  valueuom  observations  minimum  \
0  220045                   Heart Rate       bpm           859     42.0   
1  220210             Respiratory Rate  insp/min           860      0.0   
2  220277  O2 saturation pulseoxymetry         %           782     64.0   

   median  maximum  
0    91.0    160.0  
1    18.0     45.0  
2    98.0    100.0  


In [9]:
missing_summary = extract.isna().sum().rename("missing_values").to_frame()
warning_summary = extract["warning"].value_counts(dropna=False).sort_index().rename("observations").to_frame()

# Exact duplicate definition for the project resource.
duplicate_columns = [
    "subject_id",
    "stay_id",
    "charttime",
    "itemid",
    "valuenum",
]
exact_duplicates = int(extract.duplicated(subset=duplicate_columns).sum())

# Every represented ICU stay should contain all three target measurement types.
measurement_types_per_stay = extract.groupby("stay_id")["itemid"].nunique()
stays_with_all_three = int((measurement_types_per_stay == 3).sum())

print("Missing values by column:")
display(missing_summary)
print("Warning flags:")
display(warning_summary)
print(f"Exact duplicates: {exact_duplicates}")
print(
    f"Stays with all three measurement types: {stays_with_all_three} / "
    f"{extract['stay_id'].nunique()}"
)

Missing values by column:
Warning flags:
Exact duplicates: 0
Stays with all three measurement types: 133 / 133


,missing_values
subject_id,0
hadm_id,0
admission_type,0
stay_id,0
first_careunit,0
charttime,0
itemid,0
measurement_label,0
valuenum,0
valueuom,0


,observations
warning,
0.0,2500
1.0,1


In [10]:
# Summarize categories at their actual entity level.
admission_type_summary = (
    extract[["subject_id", "hadm_id", "admission_type"]].drop_duplicates()
    ["admission_type"].value_counts(dropna=False).rename_axis("admission_type")
    .rename("hospital_admissions").to_frame()
)
first_careunit_summary = (
    extract[["stay_id", "first_careunit"]].drop_duplicates()
    ["first_careunit"].value_counts(dropna=False).rename_axis("first_careunit")
    .rename("icu_stays").to_frame()
)
display(admission_type_summary)
display(first_careunit_summary)


,hospital_admissions
admission_type,
EW EMER.,58
URGENT,24
OBSERVATION ADMIT,17
SURGICAL SAME DAY ADMISSION,13
ELECTIVE,5
DIRECT EMER.,4


,icu_stays
first_careunit,
Medical Intensive Care Unit (MICU),29
Surgical Intensive Care Unit (SICU),29
Medical/Surgical Intensive Care Unit (MICU/SICU),23
Cardiac Vascular Intensive Care Unit (CVICU),19
Trauma SICU (TSICU),16
Coronary Care Unit (CCU),13
Neuro Surgical Intensive Care Unit (Neuro SICU),2
Neuro Stepdown,1
Neuro Intermediate,1


### Reproducibility checks

These assertions make accidental changes in the derivation visible. If the source release or extraction logic changes, the notebook should fail here rather than silently producing a different canonical resource.

In [11]:
expected_item_counts = {220045: 859, 220210: 860, 220277: 782}
actual_item_counts = extract["itemid"].value_counts().sort_index().to_dict()

assert len(extract) == 2501
assert extract["subject_id"].nunique() == 94
assert extract["hadm_id"].nunique() == 121
assert extract["stay_id"].nunique() == 133
assert actual_item_counts == expected_item_counts
assert extract["valuenum"].isna().sum() == 0
assert extract["valueuom"].isna().sum() == 0
assert exact_duplicates == 0
assert stays_with_all_three == 133

assert list(extract.columns) == [
    "subject_id", "hadm_id", "admission_type", "stay_id", "first_careunit",
    "charttime", "itemid", "measurement_label", "valuenum", "valueuom", "warning",
]
assert extract[["admission_type", "first_careunit"]].notna().all().all()
assert extract["admission_type"].nunique() == 6
assert extract["first_careunit"].nunique() == 9
assert extract.groupby(["subject_id", "hadm_id"])["admission_type"].nunique().eq(1).all()
assert extract.groupby("stay_id")["first_careunit"].nunique().eq(1).all()
assert extract["warning"].value_counts().to_dict() == {0: 2500, 1: 1}

print("All reproducibility checks passed.")

All reproducibility checks passed.


## 8. Save the canonical extract

The generated CSV is the bounded tabular resource that the next FAIRification stages will model semantically and transform. The original MIMIC-IV Demo remains the provenance source.

In [12]:
DERIVED_DIR.mkdir(parents=True, exist_ok=True)
output_path = DERIVED_DIR / "mimic_icu_vitals_4h.csv"

extract.to_csv(output_path, index=False, date_format="%Y-%m-%d %H:%M:%S")

sha256 = hashlib.sha256(output_path.read_bytes()).hexdigest()
print(f"Saved: {output_path.relative_to(PROJECT_ROOT)}")
print(f"Rows: {len(extract):,}")
print(f"SHA-256: {sha256}")

Saved: data/derived/mimic_icu_vitals_4h.csv
Rows: 2,501
SHA-256: 2171c7d336272e02ef835146a21dbfa8503f74451def29e50f5b87704392c120


## Result

The project now has a reproducible source-to-resource derivation:

**MIMIC-IV Demo v2.2 ICU and hospital-admission tables → documented selection rule → validated 4-hour HR/RR/SpO2 extract.**

The next project step is to document the meaning of the eleven output fields and perform the initial FAIR/planning assessment before defining the RDF semantic data model.